To load base model as well as lora parameters finetuned by SFTT, which will be tested by inference with specific inputs.  

In [2]:
import os
import pandas as pd
import torch
import re
import io
import sys
import ast
import time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
# --- 1. Configuration ---
    
# Define the device to use (e.g., 'cuda:0' for the first GPU)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using device: {device}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

Using device: cuda:0


In [4]:
# --- 2. Load finetuned Model and Tokenizer ---

# Define paths for the base model and the LoRA adapter
base_model_path = "/local/project/models--unsloth--Qwen3-4B-Base/snapshots"
lora_adapter_path = "/local/project/outputs/sftt_save_lora_v3.8"

In [5]:
from unsloth import FastLanguageModel
import torch
from peft import PeftModel
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower
# --- Load base model ---
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, #True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)
'''
# ---Load lora layer---
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)
'''
print("Base model loaded successfully with Unsloth.")      
if tokenizer.pad_token is None:        
    tokenizer.pad_token = tokenizer.eos_token 
#model = PeftModel.from_pretrained(base_model, lora_adapter_path)
#print("LoRA adapter loaded and merged successfully.")    

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-08-16 16:48:57.868469: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 16:48:57.894746: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-16 16:48:57.894772: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-16 16:48:57.895493: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-16 16:48:57.900442: I tensorflow/core/platform/cpu_feature_guar

🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 08-16 16:49:00 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 08-16 16:49:01 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.6.6: Fast Qwen3 patching. Transformers: 4.52.4. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA GeForce RTX 4090 Laptop GPU. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded successfully with Unsloth.


In [11]:
model=base_model

In [6]:
# --- 2. SYSTEM PROMPTS (Must match training) ---
code_system_prompt = \
"""You are an expert Python programmer. Your sole task is to write a self-contained Python script to solve the given computational problem.
- Your response MUST begin directly with the code block ```python and end with ```.
- Do NOT provide any text or explanation before or after the code block.
- The script must define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- Do NOT solve the problem yourself or provide any reasoning inside the script, just return the caculation.
- Analyze the problem carefully and choose the appropriate response format, which should contain non-repetitive answers.
- For example, for 'caculate the value of (1+1)', your entire response must be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

cot_code_system_prompt = \
"""You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for 'caculate the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

In [7]:
# --- 2.1 To determine which type of problem it is and then encapsulate it with respective prompt  
def select_prompt(problem: str) -> str:
    """
    Selects the appropriate system prompt based on the problem's content.
    This should roughly match the logic used to categorize data during training.
    """
    problem_lower = problem.lower()

    if "calculate the" in problem_lower:
        print("(System 01: Detected computational problem, using Code-Only prompt.)")
        problem = problem.replace("Calculate","$Calculate#")
        return code_system_prompt, problem
    
    # Keywords that suggest a need for reasoning/proof (CoT)
    proof_keywords = [
        # Original keywords
        'prove that', 'show that', 'demonstrate that', 'explain why',
        'is it true that', 'determine', 'converge', 'convergent',
        'relationship', 'what is the probability',
        # Added keywords from user examples and common math terms
        'equation', 'derivative', 'expansion', 'how many', 'what is',
        'compute the', 'simplify', 'solve for', 'express', 'theorem',
        'proof', 'show', 'derive', 'relates', 'find an', 'find the', 'calculate the'
    ]
    
    # Mathematical symbols/patterns that often appear in theoretical problems
    proof_symbols = [
        # Original symbols
        r'\\sum', r'\\int', r'\\lim', r'\\infty', r'\\binom', r'\\choose',
        r'\\prod', r'\\partial', r'\\nabla', r'\\subset', r'\\in',
        # Added symbols and patterns
        r'\\theta', r'\\sec', r'\\pi', r'\\alpha', r'\\beta', r'\\gamma',
        r'\\delta', r'\\sin', r'\\cos', r'\\tan', r'\\log',
        # Match 'ln' as a word, or '\\ln' for latex
        r'\\ln', r'\bln\b',
        r'\\sqrt',
        # Match f(x), g(x), etc.
        r'f\(x\)', r'g\(x\)', r'h\(x\)',
        r'd/dx', r'\^',
        # Corrected patterns for escaped parentheses for LaTeX e.g. \\( ... \\)
        r'\\\(' , r'\\\)',
        # Pattern for simple parentheses in computational problems e.g. (1+2)
        # This is a broad match, but problems with parens are often not simple arithmetic.
        r'\('
    ]

    if any(keyword in problem_lower for keyword in proof_keywords) or \
       any(re.search(symbol, problem) for symbol in proof_symbols):
        print("(System 02: Detected theoretical problem, using CoT+Code prompt.)")
        return cot_code_system_prompt, problem
    else:
        print("(System 03: Detected computational problem, using Code-Only prompt.)")
        return code_system_prompt, problem

In [8]:
# --- 2.2. Prepare the test questions set ---
question_set=["Calculate the value of (0x1235 XOR 0xfb67)",
              "Calculate the value of (0x36789 AND 0x67fac OR 0x209b7)",
              "Calculate the value of (1567*12 + 3**17 + ln(7))",
              "Calculate the value of (56129087+23458765)",
              "Calculate the value of (87165*33921)",
              "Calculate the value of (2765420987-127654987)",
              "Calculate the value of (76552298/7654322)",
              "Calculate the value of ((0x2a765 >> 16 ) and (0xb7 << 8))",
              #"Caculate the 32-bit 0x2a765 right cyclic shift by 16 bits.",
              "Calculate the 32-bit right cyclic shift of 0x2a765 by 16 bits.",
              #"Assume 0x2a765 is a 32-bit number, caculate the value after cyclic right-shift 16 bit ",
              "Calculate the value of (e**2 + pi*2/3 + 130)",
              "Mary has taken three tests and has a test average of 87. Her parents want her to maintain an average of at least 85. What is the lowest score that Mary can get on her next test while keeping her average at least 85?",
              "A rectangle is inscribed in a circle with an area of $16\\pi$. The area of the rectangle is $30$. Find the perimeter of the rectangle.",  
              "Given $\\sqrt{x^2+38}-\\sqrt{x^2-6}=4$ and $x$ is positive, find all possible values of $x$.",
              "What is the sum of all real numbers $x$ for which $|x^2 - 6x + 12| = 3$?",
              "Suppose that the equations $y = x^3 - 3x + 5$ and $x + 2y = 8$ intersect at the points $(x_1, y_1)$, $(x_2, y_2)$, and $(x_3, y_3)$. What is the value of $x_1 + x_2 + x_3 + y_1 + y_2 + y_3$?",
              "What is the minimum distance a fly must travel when flying from one corner to the opposite corner of a rectangular box that measures 3 feet by 5 feet by 6 feet?",
              "In a summer camp with 100 students, each student can sing, dance, or act. Some students have more than one talent, but no student has all three talents. There are 42 students who cannot sing, 65 students who cannot dance, and 29 students who cannot act. How many students have two of these talents?",
              "At his usual rowing rate, a man rows 15 miles downstream in five hours less time than it takes him to return. If he doubles his usual rowing rate, the time downstream is only one hour less than the time upstream. Find the rate of the stream's current in miles per hour.",
              "Jason has the same number of red and blue marbles. He puts them in two jars so that the ratio of red to blue marbles in jar I is 2:5 and in jar II is 9:5. If there are 84 marbles in jar I, how many marbles are there in jar II?",
              "A person is walking down an escalator moving downwards and takes a total of 26 steps to reach the bottom in 30 seconds. When running down the escalator, the same person takes a total of 34 steps and reaches the bottom in 18 seconds. How many steps are on the escalator at a time?",
              "Let $(A_{ij})$ be an $n \\times n$ matrix and $(B_{ij})$ be the cofactor matrix of $(A_{ij})$. What is the rank of $(B_{ij})$ when the rank of $(A_{ij})$ is $\\le n-2$?",
              "On a table, there are 2016 coins. Two players take turns, and in each turn, a player can take 1, 2, or 3 coins. The player who takes the last coin wins. Which player has a winning strategy?",
              "Find an equation that relates the observable distance \\(d\\) on the Earth's surface to the central angle \\(\\theta\\).",
              "Compute the derivative of \\( f(x) = e^{x \\sec(x)} \\).",
              "How many terms are in the expansion of $(a+b+c+d)^n$ (in terms of $n$)?",
              "Find the sum of the infinite series \\(9 - 3 + 1 - \\frac{1}{3} + \\frac{1}{9} + \\cdots\\).",
              "Find a formula for \\( \\binom{n}{0}^2 + \\binom{n}{1}^2 + \\binom{n}{2}^2 + \\cdots + \\binom{n}{n}^2 \\).",
              "Factor $6x^{12} + 35x^7 + 36x^2$.",
              "Study the Lebesgue measurability and integrability of the function $f:(0,1)\\rightarrow\\mathbb{R}$, which is continuous and satisfies $\\lvert f(x) \\rvert \\le \\frac{1}{\\sqrt x}$ for all $x\\in(0,1)$.",
              "Under what conditions is \\( X^T X \\) invertible?"
             ]

In [92]:
# --- 2.3. Select the question and its system prompt ---
current_question = question_set[29]
system_prompt, current_question = select_prompt(current_question)
print(current_question,"~~~~~~", system_prompt)

(System 02: Detected theoretical problem, using CoT+Code prompt.)
Under what conditions is \( X^T X \) invertible? ~~~~~~ You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for 'caculate the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.



In [93]:
# --- 2.4. Apply the Prompt (Crucial Step) ---
question = current_question
multi_task_system_prompt = system_prompt

# Construct the 'messages' list in the same format as your training data.
messages = [
    {"role": "system", "content": multi_task_system_prompt},
    {"role": "user", "content": question},
]

# Apply the chat template to format the input correctly.
# `add_generation_prompt=True` is essential for inference.
inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True, 
        padding=True, # This is important for batching, good practice for single inference
        return_tensors="pt"
).to(device)

# --- 3. Generate the Response ---
start_time = time.time()    
prompt_token_length = inputs.shape[1]
print("\nGenerating response...")
outputs = model.generate(
    input_ids=inputs, 
    max_new_tokens=2048, 
    use_cache=True,
    do_sample=True,
    temperature=0.2, # Lower the temperature to reduce randomness
    top_p=0.9
) 

outputs_without_prompt = outputs[:, prompt_token_length: ]
response_text = tokenizer.batch_decode(outputs_without_prompt, skip_special_tokens=True)[0]
print("\nFinish reasoning.")
end_time = time.time()
ref_time = end_time - start_time
print("It cost {} seconds for reference".format(ref_time))
# --- 5. Print the Cleaned Response ---
    
# Extract only the assistant's part of the response for clarity.
try:
    assistant_response = response_text.split("<|im_start|>assistant\n")[-1].strip()
except IndexError:
    assistant_response = response_text # Fallback if the template isn't found

print("\n--- Model Response ---")
print(assistant_response)
print("----------------------\n")


Generating response...

Finish reasoning.
It cost 28.44726586341858 seconds for reference

--- Model Response ---
To determine when \( X^T X \) is invertible, we need to understand the properties of the matrix \( X \). Specifically, we need to consider the rank of \( X \) and the relationship between \( X \) and its transpose \( X^T \).

### Step-by-Step Explanation

1. **Matrix Rank and Invertibility**:
   - A square matrix \( A \) is invertible if and only if it has full rank, meaning its rank is equal to the number of rows (or columns) of the matrix.
   - For a matrix \( X \) of size \( m \times n \), the rank of \( X \) is at most \( \min(m, n) \).

2. **Rank of \( X^T X \)**:
   - The rank of \( X^T X \) is equal to the rank of \( X \). This is because the rank of a product of matrices is at most the rank of each individual matrix.
   - Therefore, if \( X \) has full column rank (i.e., \( \text{rank}(X) = n \)), then \( X^T X \) will also have full rank and will be invertible.

3

# --- 4. Extract python code from the response 
pattern = r"```(?:python|py|)\s*?(.*?)```"
matches = re.findall(pattern, assistant_response, re.DOTALL)
#print matches
#matches

if matches:
    for i, python_code in enumerate(matches):
        print(f"\nExtracted Code {i+1}:")
        print(python_code.strip())
else:
    print("No python code found.")

In [13]:
# --- 5. Extract python code from the response 
#pattern = r"```(?:python|py|)\s*?(.*?)```"
pattern = r"```+(?:python|py)?\s*\n(.*?)\n```+"
matches = re.findall(pattern, assistant_response, re.DOTALL)
#print matches
#matches

if matches:
    for i, python_code in enumerate(matches):
        print(f"\nExtracted Code {i+1}:")
        print(python_code.strip())
else:
    print("No python code found.")


Extracted Code 1:
def solve():
    return 0x1235 ^ 0xfb67
print(solve())


In [14]:
def run_code(code_string: str) -> tuple[str, bool]:
    """
    Executes a string of Python code in a safe scope and captures the output.

    This function is robust and handles three common cases for LLM-generated code:
    1. Code with explicit print() statements: It captures the printed output.
    2. Code ending in an expression (e.g., '3 + 5'): It returns the value.
    3. Code ending in a statement (e.g., 'a = 5'): It captures nothing, which is correct.

    Args:
        code_string: The Python code to execute.

    Returns:
        A tuple containing:
        - The captured output or result as a string.
        - A boolean indicating if an error occurred (True if error, False otherwise).
    """
    output_buffer = io.StringIO()
    original_stdout = sys.stdout
    scope = {}

    # Ensure stdout is restored even if errors occur
    try:
        # Redirect stdout to capture prints
        sys.stdout = output_buffer
        last_expr = None

        # Use the 'ast' module to find and evaluate the last expression
        try:
            tree = ast.parse(code_string.strip())
            if tree.body and isinstance(tree.body[-1], ast.Expr):
                # If the last node is an expression, wrap it in a print() call.
                # This makes code like 'a=5; a+10' correctly output '15'.
                last_expr_node = tree.body.pop()
                last_expr = ast.unparse(last_expr_node).strip()

                # Create the new print node
                print_node = ast.Expr(
                    value=ast.Call(
                        func=ast.Name(id='print', ctx=ast.Load()),
                        args=[last_expr_node.value],
                        keywords=[]
                    )
                )
                tree.body.append(ast.fix_missing_locations(print_node))

            # Compile the (potentially modified) code and execute it
            exec(compile(tree, '<string>', 'exec'), scope)
            #last_result = eval(last_expr, scope)
        except (SyntaxError, Exception) as e:
            # If parsing or execution fails, print the error to the real stderr
            # and also capture it in our return value.
            print(f"Error executing code: {e}", file=sys.stderr)
            return f"Error: {e}", True

    finally:
        # ALWAYS restore the original stdout
        sys.stdout = original_stdout

    # Get the complete output from the buffer and clean it up
    captured_output = output_buffer.getvalue().strip()
    if last_expr:
        if ("print" in last_expr):
            if (eval(last_expr, scope)==None):
                captured_output = output_buffer.getvalue().strip().replace("\nNone","")
    return captured_output, False

In [94]:
# --- 6 The final response with result of code execution ---

pattern = r"```+(?:python|py)?\s*\n(.*?)\n```+"
matches = re.findall(pattern, assistant_response, re.DOTALL)
revised_content = assistant_response
if matches:
    for i, code_block in enumerate(matches):
        code_block = code_block.strip()
        if not code_block:
            continue

        print(f"\n--- Executing Block {i+1} ---")
        print(code_block)

        #This one line runs the code and gets the result
        result, had_error = run_code(code_block)
            
        print(f"Result: {result}")
        print(f"Error Occurred: {had_error}")
            
        # Create the text to insert back into the model's response
        if had_error:
            # Use a distinct marker for errors
            insert_text = f"```python\n{code_block}\n\n---[Code Execution Error]---\n{result}\n```"
        else:
            # Use a clean marker for successful output
            insert_text = f"```python\n{code_block}\n\n---[Code Output]---\n{result}\n```"

        # Replace the original code block with the annotated version
        original_block_in_markdown = f"```python\n{code_block}\n```"
        revised_content = revised_content.replace(original_block_in_markdown, insert_text, 1)

print("\n\n--- Final Revised Content ---")
print(revised_content)


--- Executing Block 1 ---
import numpy as np

def solve():
    # Define a matrix X
    X = np.array([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])
    
    # Compute X^T X
    XTX = np.dot(X.T, X)
    
    # Check if X has full column rank
    rank_X = np.linalg.matrix_rank(X)
    n = X.shape[1]  # Number of columns in X
    
    # Determine if X^T X is invertible
    is_invertible = rank_X == n
    
    return is_invertible

# Call the solve function
print(solve())
False
Result: False
Error Occurred: False


--- Final Revised Content ---
To determine when \( X^T X \) is invertible, we need to understand the properties of the matrix \( X \). Specifically, we need to consider the rank of \( X \) and the relationship between \( X \) and its transpose \( X^T \).

### Step-by-Step Explanation

1. **Matrix Rank and Invertibility**:
   - A square matrix \( A \) is invertible if and only if it has full rank, meaning its rank is equal to the number of rows (or columns)

In [1]:
# Release GPU resource 
import torch
from numba import cuda
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print("Releasing GPU memory (cleanup)...")
    cuda.get_current_device().reset()
    print("GPU memory released.")

import gc
gc.collect()

Releasing GPU memory (cleanup)...
GPU memory released.


23

In [1]:
# To kill the subprocess of PID directly to empty the memory of GPU in case of failure after restarting kernel 
#!kill -9 9190

In [1]:
!nvidia-smi

Sat Aug 16 16:48:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.51                 Driver Version: 561.19         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              5W /  135W |     236MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----